# <h1 style="font-family: Trebuchet MS; padding: 20px; font-size: 40px; color: #FFFFFF; text-align: center; line-height: 0.55;background-color: #FFA500"><b>Titanic dataset</b><br></h1>

### Introduction :

The Titanic dataset is a classic and widely used dataset in the field of machine learning and data analysis. It provides valuable insights into the survival rates of passengers aboard the ill-fated RMS Titanic, which sank on its maiden voyage in 1912 after colliding with an iceberg.

This dataset contains information about various passengers such as their age, gender, class, ticket fare, cabin, and survival status. It serves as an excellent resource for exploring the factors that influenced the survival of individuals on the Titanic and for developing predictive models using machine learning algorithms.

The main objective when working with the Titanic dataset is typically to predict whether a given passenger survived or perished based on the available features. This task is a classic example of binary classification, where the target variable is the survival status (0 for didn't survive, and 1 for survived).

Machine learning techniques can be applied to the Titanic dataset to build predictive models that can accurately classify passengers as survivors or non-survivors based on their characteristics. By analyzing the dataset and using appropriate algorithms, we can uncover patterns and relationships between the features and the target variable, thus creating a model capable of making predictions on unseen data.

In conclusion, the Titanic dataset presents an exciting opportunity to apply machine learning techniques to a real-world problem and gain insights into the factors that influenced survival on the ill-fated ship. By leveraging this dataset, researchers and practitioners can develop and refine machine learning models while deepening their understanding of feature engineering and predictive analytics.


# <center><div style="font-family: Trebuchet MS; background-color: #FFA500; color: #FFFFFF; padding: 12px; line-height: 1;">Dataset Information</div></center>

### Import the Necessary Libraries :

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
pd.options.display.float_format = '{:.2f}'.format
import warnings
warnings.filterwarnings('ignore')
from tqdm import tqdm


from sklearn.preprocessing import LabelEncoder

In [ ]:
data = pd.read_csv('titanic.csv')
data.head()

In [ ]:
data.drop(columns=['Ticket', 'Name', 'PassengerId'], inplace=True)

### Data Info :

In [ ]:
data.columns

In [ ]:
data.shape

In [ ]:
data.info()

In [ ]:
data.dtypes

### Missing Data Analysis

In [ ]:
sns.heatmap(data.isnull(),cmap = 'magma',cbar = False)

In [ ]:
data.isna().sum().sort_values(ascending=False).to_frame().style.set_properties(**{"background-color": "#40E0D0",
                                                                                  "color":"black","border": "1.5px solid black"})

In [ ]:
missing = data.isnull().sum()
missing = missing[missing>0]
missing.sort_values(inplace=True, ascending=False)
ax = missing.plot(kind='bar')
ax.set_alpha(0.8)
ax.set_title("Barplot of Variables and Frequency of Missing Entries")
ax.set_ylabel("Frequency")
ax.set_xlabel('Variable Name')


for i in ax.patches:

     ax.text(i.get_x() + 0.15, i.get_height() + 10, \
            str(round((i.get_height()/len(data))*100, 1))+'%', fontsize=8,
                color='dimgrey')

### Handling Missing Values

**`Cabin`** - The Cabin column contains information about the cabins assigned to passengers. However, due to the large number of missing values in this column, it becomes challenging to impute or infer the missing data accurately. Removing the Cabin column simplifies the dataset, allowing us to focus on the remaining variables that have more complete information.


**`Age`** - To address the missing values in the "Age" column of the Titanic dataset, we can develop a function that takes into account the corresponding passenger class ("Pclass") of each individual. The "Pclass" column represents the socio-economic status of the passengers, ranging from 1 (upper class) to 3 (lower class).

The function can be designed to impute the missing age values based on the mean or median age of passengers within the same passenger class. By utilizing this approach, we can leverage the correlation between socio-economic status and age to estimate the missing values more accurately.


**``Embarked``** - To address the missing values in the "Embarked" column of the Titanic dataset, we can employ a simple strategy of imputing the missing values with the most frequent value observed in the column. The "Embarked" column represents the port of embarkation for each passenger, with possible values being "C" (Cherbourg), "Q" (Queenstown), and "S" (Southampton).


In [ ]:
data['Embarked'].value_counts()

In [ ]:
data['Embarked'].replace(np.nan, 'S', inplace=True)

In [ ]:
print(round(data[data['Pclass']==1]['Age'].mean(), 2))
print(round(data[data['Pclass']==2]['Age'].mean(), 2))
print(round(data[data['Pclass']==3]['Age'].mean(), 2))

In [ ]:
def impute_age(cols):
    Age = cols[0]
    Pclass = cols[1]

    if pd.isnull(Age):

        if Pclass == 1:
            return 38

        elif Pclass == 2:
            return 30

        else:
            return 25

    else:
        return Age

In [ ]:
data['Age'] = data[['Age','Pclass']].apply(impute_age,axis=1)

In [ ]:
data.drop(columns=['Cabin'], inplace=True)

In [ ]:
sns.heatmap(data.isnull(),cmap = 'magma',cbar = False)

In [ ]:
data.describe(include='object').T.style.set_properties(**{"background-color": "#40E0D0",
                                                          "color":"black","border": "1.5px solid black"})

In [ ]:
data.describe(include=['int', 'float']).T.style.set_properties(**{"background-color": "#40E0D0",
                                                          "color":"black","border": "1.5px solid black"})

In [ ]:
yes = data[data['Survived'] == 1].describe().T
no = data[data['Survived'] == 0].describe().T
colors = ['#DE3163', '#40E0D0']

fig,ax = plt.subplots(nrows = 1,ncols = 2,figsize = (5,5))
plt.subplot(1,2,1)
sns.heatmap(yes[['mean']],annot = True,cmap = colors,linewidths = 0.4,linecolor = 'black',cbar = False,fmt = '.2f',)
plt.title('Survived')

plt.subplot(1,2,2)
sns.heatmap(no[['mean']],annot = True,cmap = colors,linewidths = 0.4,linecolor = 'black',cbar = False,fmt = '.2f')
plt.title('No Survived')
plt.tight_layout(pad=3)

- **Mean** values of all the features for cases of **Survived** and **No Survived**.

# <center><div style="font-family: Trebuchet MS; background-color: #FFA500; color: #FFFFFF; padding: 12px; line-height: 1;">Exploratory Data Analysis</div></center>

### Dividing features into Numerical and Categorical :

In [ ]:
col = list(data.columns)
categorical_features = []
numerical_features = []
for i in col:
    if len(data[i].unique()) > 6:
        numerical_features.append(i)
    else:
        categorical_features.append(i)

print('Categorical Features :',*categorical_features)
print('Numerical Features :',*numerical_features)

data1 = data.copy(deep=True)

- Here, categorical features are defined if the the attribute has less than 6 unique elements else it is a numerical feature.
- Typical approach for this division of features can also be based on the datatypes of the elements of the respective attribute.

In [ ]:
data1.dtypes

In [ ]:
le = LabelEncoder()
text_data_features = ['Sex', 'Embarked']
l3 = []; l4 = [];
print('Label Encoder Transformation')
for i in tqdm(text_data_features):
    data1[i] = le.fit_transform(data1[i])
    l3.append(list(data1[i].unique())); l4.append(list(le.inverse_transform(data1[i].unique())))
    print(i,' : ',data1[i].unique(),' = ',le.inverse_transform(data1[i].unique()))

In [ ]:
tf1 = {}
for i in range(len(text_data_features)):
    tf1[text_data_features[i]] = {}
    for j,k in zip(l3[i],l4[i]):
        tf1[text_data_features[i]][j] = k

tf1['Survived'] = {0 : 'No Survived', 1 : 'Survived'}
tf1['Pclass'] = {1 : 'Upper', 2 : 'Middle', 3 : 'Lower'}
tf1

### Target Variable Visualization (Survived) :

In [ ]:
l = list(data1['Survived'].value_counts())
circle = [l[0] / sum(l) * 100,l[1] / sum(l) * 100]

fig = plt.subplots(nrows = 1,ncols = 2,figsize = (20,5))
plt.subplot(1,2,1)
plt.pie(circle,labels = ['No Survived','Survived'],autopct='%1.1f%%',startangle = 90,explode = (0.1,0),colors = colors,
       wedgeprops = {'edgecolor' : 'black','linewidth': 1,'antialiased' : True})
plt.title('present of placed (%)')

plt.subplot(1,2,2)
ax = sns.countplot(x='Survived',data = data1, palette = colors,edgecolor = 'black')
for rect in ax.patches:
    ax.text(rect.get_x() + rect.get_width() / 2, rect.get_height() + 2, rect.get_height(), horizontalalignment='center', fontsize = 11)
ax.set_xticklabels(['No Survived','Survived'])
plt.title('Number of Survived')
plt.show()

### Numerical Features :

#### Distribution of Numerical Features :

In [ ]:
fig, ax = plt.subplots(nrows = 2,ncols = 2,figsize = (10,6))
plt.subplots_adjust(wspace=0.2, hspace=0.5)

for i in range(len(numerical_features)):
    plt.subplot(2,2,i+1)
    sns.distplot(data1[numerical_features[i]],color = colors[0])
    title = 'Distribution : ' + numerical_features[i]
    plt.title(title)
plt.show()

### Numerical Features vs Target Variable (Survived) :

In [ ]:
fig, ax = plt.subplots(nrows = 4,ncols = 1,figsize = (10,25))
plt.subplots_adjust(wspace=0.1, hspace=1)



for i in range(len(numerical_features)):
    plt.subplot(4,1,i+1)
    ax = sns.countplot(x=numerical_features[i],data = data1,hue = "Survived",palette = colors,edgecolor = 'black')
    # ax.set_xticklabels([tf1[numerical_features2[i]][j] for j in sorted(data1[numerical_features2[i]].unique())])
    plt.legend(['No Survived', 'Survived'] ,loc = 'upper right')
    title = numerical_features[i] + ' w.r.t Survived'
    plt.title(title)

In [ ]:
data1['Age_group'] = [ int(i / 2) for i in data1['Age']]
data1['Fare_group'] = [ int(i / 2) for i in data1['Fare']]

In [ ]:
fig, ax = plt.subplots(nrows = 2,ncols = 1,figsize = (10,15))
plt.subplots_adjust(wspace=0.1, hspace=0.3)
group_numerical_features = [i + '_group' for i in ['Age','Fare']]

for i in range(len(group_numerical_features)):
    plt.subplot(2,1,i+1)
    sns.countplot(x=group_numerical_features[i],data = data1,hue = "Survived",palette = colors,edgecolor = 'black')
    plt.legend(['No Survived', 'Survived'] ,loc = 'upper right')
    title = group_numerical_features[i] + ' w.r.t Survived'
    plt.title(title)

In [ ]:
data1 = data1.drop(columns=[ 'Age_group', 'Fare_group'])

### Categorical Features :

#### Distribution of Categorical Features :

In [ ]:
categorical_features.remove('Survived')

In [ ]:
fig = plt.subplots(nrows = 3,ncols = 1,figsize = (10,20))
plt.subplots_adjust(wspace=0.2, hspace=0.5)
for i in range(3):
    plt.subplot(3,1,i+1)
    ax = sns.countplot(x=categorical_features[i],data = data1,hue = "Survived",palette = colors,edgecolor = 'black')
    for rect in ax.patches:
        ax.text(rect.get_x() + rect.get_width() / 2, rect.get_height() + 2, rect.get_height(), horizontalalignment='center', fontsize = 8)
    ax.set_xticklabels([tf1[categorical_features[i]][j] for j in sorted(data1[categorical_features[i]].unique())])
    plt.legend(['No Survived', 'Survived'], loc = 'upper right')
    title = categorical_features[i] + ' w.r.t Survived'
    plt.xticks(rotation=45)
    plt.title(title)

### Categorical Features vs Cases of Survived :

In [ ]:
Sex  = data1[data1['Survived'] == 1]['Sex'].value_counts()
Sex = [Sex[0] / sum(Sex) * 100, Sex[1] / sum(Sex) * 100]

Pclass  = data1[data1['Survived'] == 1]['Pclass'].value_counts()
Pclass = [pclass[1] / sum(Pclass) * 100, Pclass[2] / sum(Pclass) * 100, Pclass[3] / sum(Pclass)]

Embarked  = data1[data1['Survived'] == 1]['Embarked'].value_counts()
Embarked = [Embarked[0] / sum(Embarked) * 100, Embarked[1] / sum(Embarked) * 100, Embarked[2] / sum(Embarked) * 100]

In [ ]:
l1 = [Pclass, Sex, Embarked]

ax,fig = plt.subplots(nrows = 1,ncols = 2,figsize = (10,10))
plt.subplots_adjust(wspace=0.5, hspace=0.5)
for i in range(2):
    if len(l1[i]) == 2:
        plt.subplot(1,2,i + 1)
        plt.pie(l1[i],labels = [tf1[categorical_features[i]][j] for j in sorted(data1[data1['Survived'] == 1][categorical_features[i]].unique())],autopct='%1.1f%%',startangle = 90,explode = (0.1,0),colors = colors,
               wedgeprops = {'edgecolor' : 'black','linewidth': 1,'antialiased' : True})
        plt.title(categorical_features[i])
    else:
        plt.subplot(1,2,i + 1)
        plt.pie(l1[i],labels = [tf1[categorical_features[i]][j] for j in sorted(data1[data1['Survived'] == 1][categorical_features[i]].unique())],autopct='%1.1f%%',startangle = 90,explode = (0.1,0,0.1),colors = colors,
               wedgeprops = {'edgecolor' : 'black','linewidth': 1,'antialiased' : True})
        plt.title(categorical_features[i])

ax,fig = plt.subplots(nrows = 1,ncols = 1,figsize = (5,5))
for i in range(-1,-2,-1):
    if len(l1[i]) == 3:
        plt.subplot(1,1,-i)
        plt.pie(l1[i],labels = [tf1[categorical_features[i]][j] for j in sorted(data1[data1['Survived'] == 1][categorical_features[i]].unique())],autopct='%1.1f%%',startangle = 90,explode = (0.1,0, 0.1),colors = colors,
               wedgeprops = {'edgecolor' : 'black','linewidth': 1,'antialiased' : True})
        plt.title(categorical_features[i])

### Categorical features vs Numerical features w.r.t Target variable(Survived) :

#### Sex vs Numerical Features :

In [ ]:
fig = plt.subplots(nrows = 2,ncols = 2,figsize = (20,15))
plt.subplots_adjust(wspace=0.3, hspace=0.8)
for i in range(len(numerical_features)):
    plt.subplot(2,2,i+1)
    ax = sns.boxplot(x = 'Sex',y = numerical_features[i],data = data1,hue = 'Survived',palette = colors)
    ax.set_xticklabels([tf1['Sex'][j] for j in sorted(data1['Sex'].unique())])
    title = numerical_features[i] + ' vs Sex'
    plt.legend(['No Survived','Survived'], loc = 'upper right')
    plt.xticks(rotation=45)
    plt.title(title)

#### Pclass vs Numerical Features :

In [ ]:
fig = plt.subplots(nrows = 2,ncols = 2,figsize = (20,15))
plt.subplots_adjust(wspace=0.3, hspace=0.8)
for i in range(len(numerical_features)):
    plt.subplot(2,2,i+1)
    ax = sns.boxplot(x = 'Pclass',y = numerical_features[i],data = data1,hue = 'Survived',palette = colors)
    ax.set_xticklabels([tf1['Pclass'][j] for j in sorted(data1['Pclass'].unique())])
    title = numerical_features[i] + ' vs Pclass'
    plt.legend(['No Survived','Survived'], loc = 'upper right')
    plt.xticks(rotation=45)
    plt.title(title)


#### Embarked vs Numerical Features :

In [ ]:
fig = plt.subplots(nrows = 2,ncols = 2,figsize = (20,15))
plt.subplots_adjust(wspace=0.3, hspace=0.8)
for i in range(len(numerical_features)):
    plt.subplot(2,2,i+1)
    ax = sns.boxplot(x = 'Embarked',y = numerical_features[i],data = data1,hue = 'Survived',palette = colors)
    ax.set_xticklabels([tf1['Embarked'][j] for j in sorted(data1['Embarked'].unique())])
    title = numerical_features[i] + ' vs Embarked'
    plt.legend(['No Survived','Survived'], loc = 'upper right')
    plt.xticks(rotation=45)
    plt.title(title)

# <center><div style="font-family: Trebuchet MS; background-color: #FFA500; color: #FFFFFF; padding: 12px; line-height: 1;">Feature Engineering</div></center>

### Correlation Matrix :

In [ ]:
plt.figure(figsize = (15,5))
sns.heatmap(data1.corr(),cmap = colors,annot = True)

In [ ]:
plt.figure(figsize=(12,5))
corr = data1.corrwith(data1['Survived']).sort_values(ascending = False).to_frame()
corr.columns = ['Survived']
sns.heatmap(corr,annot = True,cmap = 'RdYlGn',linewidths = 0.4,linecolor = 'black');
plt.title('Correlation with survived')


### Data Scaling :

In [ ]:
from sklearn.preprocessing import MinMaxScaler,StandardScaler
mms = MinMaxScaler() # Normalization
ss = StandardScaler() # Standardization

# Normalization
data1['Age'] = mms.fit_transform(data1[['Age']])
data1['SibSp'] = mms.fit_transform(data1[['SibSp']])
data1['Parch'] = mms.fit_transform(data1[['Parch']])
data1['Fare'] = mms.fit_transform(data1[['Fare']])

# Standardization
data1['Pclass'] = ss.fit_transform(data1[['Pclass']])
data1['Sex'] = ss.fit_transform(data1[['Sex']])
data1['Embarked'] = ss.fit_transform(data1[['Embarked']])

data1.head()

# <center><div style="font-family: Trebuchet MS; background-color: #FFA500; color: #FFFFFF; padding: 12px; line-height: 1;">Modeling</div></center>

In [ ]:
x = data1.drop(columns=['Survived'])
y = data1['Survived']

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import precision_recall_curve

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=101)

In [ ]:
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

In [ ]:
def model(classifier,x_train,y_train,x_test,y_test):

    classifier.fit(x_train,y_train)
    prediction = classifier.predict(x_test)
    cv = RepeatedStratifiedKFold(n_splits = 10,n_repeats = 3,random_state = 1)
    print("Cross Validation Score : ",'{0:.2%}'.format(cross_val_score(classifier,x_train,y_train,cv = cv,scoring = 'roc_auc').mean()))
    print("ROC_AUC Score : ",'{0:.2%}'.format(roc_auc_score(y_test,prediction)))


def model_evaluation(classifier,x_test,y_test):

    # Confusion Matrix
    cm = confusion_matrix(y_test,classifier.predict(x_test))
    names = ['True Neg','False Pos','False Neg','True Pos']
    counts = [value for value in cm.flatten()]
    percentages = ['{0:.2%}'.format(value) for value in cm.flatten()/np.sum(cm)]
    labels = [f'{v1}\n{v2}\n{v3}' for v1, v2, v3 in zip(names,counts,percentages)]
    labels = np.asarray(labels).reshape(2,2)
    sns.heatmap(cm,annot = labels,cmap = colors,fmt ='')

    # Classification Report
    print(classification_report(y_test,classifier.predict(x_test)))




def plot_roc_curve(y_true, y_scores):
    # Calculate the false positive rate (FPR) and true positive rate (TPR)
    fpr, tpr, _ = roc_curve(y_true, y_scores)

    # Calculate the area under the ROC curve (AUC)
    auc = roc_auc_score(y_true, y_scores)

    # Plot the ROC curve
    plt.figure()
    plt.plot(fpr, tpr, label='ROC curve (AUC = {:.2f})'.format(auc))
    plt.plot([0, 1], [0, 1], 'k--')  # Diagonal line (random classifier)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate (FPR)')
    plt.ylabel('True Positive Rate (TPR)')
    plt.title('Receiver Operating Characteristic (ROC) Curve')
    plt.legend(loc='lower right')
    plt.show()

### 1. XGBoostClassifier :

In [ ]:
classifier_xgb = XGBClassifier(learning_rate= 0.01,max_depth = 3,n_estimators = 1000)

In [ ]:
model(classifier_xgb,x_train,y_train,x_test,y_test)
model_evaluation(classifier_xgb,x_test,y_test)

### 2. LogisticRegression :

In [ ]:
classifier_lr = LogisticRegression(random_state = 0,C=10,penalty= 'l2')

In [ ]:
model(classifier_lr,x_train,y_train,x_test,y_test)
model_evaluation(classifier_lr,x_test,y_test)

### 3. Support Vector Machine

In [ ]:
classifier_svc = SVC(kernel = 'linear',C = 0.1)

In [ ]:
model(classifier_svc,x_train,y_train,x_test,y_test)
model_evaluation(classifier_svc,x_test,y_test)

### 4. DecisionTreeClassifier :

In [ ]:
classifier_tree = DecisionTreeClassifier(random_state=101, criterion='entropy', max_depth=4, min_samples_leaf=3)

In [ ]:
model(classifier_tree,x_train,y_train,x_test,y_test)
model_evaluation(classifier_tree,x_test,y_test)

### 5. KNeighborsClassifier :

In [ ]:
classifier_knn = KNeighborsClassifier(leaf_size = 1, n_neighbors = 3,p = 1)

In [ ]:
model(classifier_knn,x_train,y_train,x_test,y_test)
model_evaluation(classifier_knn,x_test,y_test)

### 6. RandomForestClassifier :

In [ ]:
classifier_rf = RandomForestClassifier(max_depth = 4,random_state = 0)

In [ ]:
model(classifier_rf,x_train,y_train,x_test,y_test)
model_evaluation(classifier_rf,x_test,y_test)

### ML Alogrithm Results Table :



|List|ML Algorithm|Cross Validation Score|ROC AUC Score|
|-|-|-|-|
|1|XGB Classifier|86.70%|80.09%|
|2|Logistic Regression|84.64%|81.23%|
|3|Support Vector Classifier|81.18%|78.19%|
|4|Tree Classifier|85.22%|77.35%|
|5|KNN Regression|82.10%|82.83%|
|6|Random Forest Classifier|86.58%|79.11%|


